# AcesAPI: a season of tennis in five minutes

This notebook loads the free AcesAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: AcesAPI (acesapi.com), under the [AcesAPI Sample Data Licence](https://github.com/sports-data-hq/acesapi-mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/sports-data-hq/acesapi-mcp/releases/download/season-pack/acesapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('atp', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## Year-end ranking

In [ ]:
t('atp', 'rankings').head(10)

## Our Elo, overall and by surface

In [ ]:
pl = t('atp', 'players')
pl.sort_values('elo_end_of_season', ascending=False)[['player', 'wins', 'losses', 'titles', 'elo_end_of_season', 'elo_hard', 'elo_clay', 'elo_grass']].head(10)

## Clay specialists: clay Elo minus hard-court Elo, minimum 20 matches

In [ ]:
pl = t('atp', 'players')
pl[pl.matches >= 20].assign(clay_edge=lambda d: d.elo_clay - d.elo_hard).sort_values('clay_edge', ascending=False)[['player', 'elo_clay', 'elo_hard', 'clay_edge']].head(10)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the AcesAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=acesapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/tennis/get_rankings', data=b'{}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the tennis skills from https://github.com/sports-data-hq/acesapi-mcp.